In [16]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [17]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [18]:
def generate_random_bit(n):
    qc = QuantumCircuit(n,n)

    #set to |+>
    for i in range(n):
      qc.h(i)
    #measure
    qc.measure(range(n),range(n))

    #run
    backend = BasicSimulator()
    compiled_circuit = transpile(qc, backend)
    job = backend.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    bitstring = list(counts.keys())[0]
    return [int(bit) for bit in bitstring]

In [19]:
def prepare_bits(bit,basis):
  qc = QuantumCircuit(1,1)
  if bit == 1:
    qc.x(0)

  if basis == 1:
    qc.h(0)
  return qc

In [20]:
def measure_qubit(qc,basis):
  if basis == 1:
    qc.h(0)

  #measure
  qc.measure(0,0)

  #run
  backend = BasicSimulator()
  compiled_circuit = transpile(qc, backend)
  job = backend.run(compiled_circuit, shots=1)
  result = job.result()
  counts = result.get_counts()

  bitstring = list(counts.keys())[0]
  return int(bitstring)

In [21]:
def create_shared_key(a_bits,a_bases,b_bases,b_results):
  a_key = []
  b_key = []
  matching_bit_pos = []

  #iterate against a_bits, if both a and b bases match, add to a and b keys and matching bit pos
  for i in range(len(a_bits)):
    if a_bases[i] == b_bases[i]:
      a_key.append(a_bits[i])
      b_key.append(b_results[i])
      matching_bit_pos.append(i)
  return a_key,b_key,matching_bit_pos

In [22]:

def a_prepare(n):
    a_bits  = generate_random_bit(n)   # A's random key bits
    a_bases = generate_random_bit(n)   # A's random encoding bases
    qubits  = [prepare_bits(a_bits[i], a_bases[i]) for i in range(n)]
    return a_bits, a_bases, qubits

def b_measure(qubits):
    n = len(qubits)
    b_bases   = generate_random_bit(n)   # B's random measurement bases
    b_results = [measure_qubit(qubits[i], b_bases[i]) for i in range(n)]
    return b_bases, b_results

def attacker_intercept(qubits):
    n = len(qubits)
    attacker_bases   = generate_random_bit(n)   # Attacker's random bases
    attacker_results = []
    tampered_qubits  = []

    for i in range(n):
        # Measure A's qubit
        measured_bit = measure_qubit(qubits[i], attacker_bases[i])
        attacker_results.append(measured_bit)

        # Re-prepare a fresh qubit to forward to B
        new_qubit = prepare_bits(measured_bit, attacker_bases[i])
        tampered_qubits.append(new_qubit)

    return attacker_bases, attacker_results, tampered_qubits

In [23]:
DETECTION_THRESHOLD = 0.10

def detect_attack(a_key, b_key):
    if len(a_key) == 0:
        return 0.0, False
    errors = sum(a != b for a, b in zip(a_key, b_key))
    qber = errors / len(a_key)
    return qber, qber > DETECTION_THRESHOLD

In [24]:
def show_results(a_bits, a_bases, attacker_bases, attacker_results,
                 b_bases, b_results, a_key, b_key, matching):
    n = len(a_bits)

    print("\n" + "="*74)
    print("BB84 TRANSMISSION  (with Attacker)")
    print("="*74)

    rows = {
        "Position"        : [f"{i:2d}"               for i in range(n)],
        "A bit"           : [f" {b}"                 for b in a_bits],
        "A basis"         : [f" {'X' if b else 'Z'}" for b in a_bases],
        "Attacker basis"  : [f" {'X' if b else 'Z'}" for b in attacker_bases],
        "Attacker result" : [f" {r}"                 for r in attacker_results],
        "B basis"         : [f" {'X' if b else 'Z'}" for b in b_bases],
        "B result"        : [f" {r}"                 for r in b_results],
        "Match A=B"       : [" ✓" if i in matching else " ✗" for i in range(n)],
    }

    for label, values in rows.items():
        print(f"{label:17}", " ".join(values))

    print("\n" + "="*74)
    print("SIFTED KEY")
    print("="*74)
    print(f"\nA key : {a_key}")
    print(f"B key : {b_key}")

    errors = sum(a != b for a, b in zip(a_key, b_key))
    if errors == 0:
        print("\n✓ Keys match — no errors in sifted key.")
    else:
        print(f"\n✗ {errors} error(s) found in sifted key!")

In [25]:
def bb84_with_attacker(n):
    # ── A: prepare qubits ─────────────────────────────────────────────────────
    print("[A]         Generating random bits and bases, preparing qubits ...")
    a_bits, a_bases, qubits = a_prepare(n)

    # ── ATTACKER: intercept and resend ────────────────────────────────────────
    print("[ATTACKER]  Intercepting qubits, measuring, re-sending to B ...")
    attacker_bases, attacker_results, tampered = attacker_intercept(qubits)

    # ── B: measure received (tampered) qubits ─────────────────────────────────
    print("[B]         Measuring received qubits ...")
    b_bases, b_results = b_measure(tampered)

    # ── A + B: sifting over public channel ────────────────────────────────────
    print("[A + B]     Sifting — comparing bases publicly ...")
    a_key, b_key, matching = create_shared_key(a_bits, a_bases, b_bases, b_results)

    # ── Display full transmission table ───────────────────────────────────────
    show_results(a_bits, a_bases, attacker_bases, attacker_results,
                 b_bases, b_results, a_key, b_key, matching)

    # ── A + B: compute QBER and check for attacker ────────────────────────────
    qber, attack_detected = detect_attack(a_key, b_key)
    key_len = len(a_key)
    errors  = sum(a != b for a, b in zip(a_key, b_key))

    print("\nSTATISTICS")
    print("="*74)
    print(f"Qubits sent           : {n}")
    print(f"Matching bases (A=B)  : {len(matching)} ({len(matching)/n*100:.1f}%)")
    print(f"Sifted key length     : {key_len} bits")
    print(f"Errors in sifted key  : {errors}")
    print(f"QBER                  : {qber*100:.2f}%")
    print(f"Detection threshold   : {DETECTION_THRESHOLD*100:.0f}%")
    print("="*74)

    if attack_detected:
        print("⚠️  ATTACK DETECTED — QBER exceeds threshold. Key exchange aborted!")
    else:
        print("✓  No attack detected (QBER within acceptable range).")
        print("   Note: with small n this can occasionally occur by chance.")
    print("="*74)

    correct_attacker = sum(attacker_bases[i] == a_bases[i] for i in range(n))
    print(f"\n[ATTACKER STATS] Correct basis guesses : {correct_attacker}/{n} ({correct_attacker/n*100:.1f}%)")
    print(f"[ATTACKER STATS] Attack was {'DETECTED' if attack_detected else 'NOT detected'}.")

    return a_key, b_key, attack_detected

In [26]:
a_key, b_key, detected = bb84_with_attacker(20)

[A]         Generating random bits and bases, preparing qubits ...
[ATTACKER]  Intercepting qubits, measuring, re-sending to B ...
[B]         Measuring received qubits ...
[A + B]     Sifting — comparing bases publicly ...

BB84 TRANSMISSION  (with Attacker)
Position           0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19
A bit              0  0  1  0  1  1  1  1  0  1  0  1  0  0  1  0  0  1  1  1
A basis            X  Z  Z  Z  X  X  Z  X  Z  Z  Z  X  X  X  X  X  X  Z  X  X
Attacker basis     Z  X  X  Z  X  X  Z  X  X  Z  Z  Z  X  X  X  Z  Z  X  Z  X
Attacker result    1  1  1  0  1  1  1  1  1  1  0  0  0  0  1  1  1  0  1  1
B basis            Z  X  X  Z  Z  X  X  Z  X  X  Z  Z  X  X  X  Z  Z  X  X  Z
B result           1  1  1  0  1  1  0  1  1  1  0  0  0  0  1  1  1  0  0  0
Match A=B          ✗  ✗  ✗  ✓  ✗  ✓  ✗  ✗  ✗  ✗  ✓  ✗  ✓  ✓  ✓  ✗  ✗  ✗  ✓  ✗

SIFTED KEY

A key : [0, 1, 0, 0, 0, 1, 1]
B key : [0, 1, 0, 0, 0, 1, 0]

✗ 1 error(s) found in sifted key!

STATIST